# 02 构建骑行事件分析表

本 Notebook 基于 01 中对原始数据的理解，进一步构建清洗后的骑行事件分析表 `trip_base`。

处理重点包括：读取并合并原始月度数据、转换时间字段、计算骑行时长、过滤非2024年7月自然月记录、剔除异常骑行时长，并新增日期、小时、星期、工作日/周末、时段、用户类型和站点分析样本标记等字段。

`trip_base` 仍保持“一行一次骑行事件”的基础粒度，后续需求规律、站点失衡和用户类型差异分析都基于该表展开。

## 1 导入依赖并设置项目路径

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path
import zipfile

DATA_RAW_DIR = Path("../data_raw")
DATA_CLEAN_DIR = Path("../data_clean")
OUTPUT_DIR = Path("../outputs")

DATA_CLEAN_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

zip_path = DATA_RAW_DIR / "202407-citibike-tripdata.zip"

## 2 查看压缩包内部文件结构

In [15]:
with zipfile.ZipFile(zip_path, "r") as z:
    file_list = z.namelist()

file_list[:10]

['202407-citibike-tripdata_2.csv',
 '202407-citibike-tripdata_3.csv',
 '202407-citibike-tripdata_1.csv',
 '202407-citibike-tripdata_4.csv',
 '202407-citibike-tripdata_5.csv']

## 3 读取并合并月度骑行数据

In [16]:
dtype_map = {
    "ride_id": "string",
    "rideable_type": "string",
    "start_station_name": "string",
    "start_station_id": "string",
    "end_station_name": "string",
    "end_station_id": "string",
    "member_casual": "string",
}

dfs = []

with zipfile.ZipFile(zip_path, "r") as z:
    csv_files = [name for name in z.namelist() if name.endswith(".csv")]
    
    for csv_file in csv_files:
        with z.open(csv_file) as f:
            temp = pd.read_csv(
                f,
                dtype=dtype_map,
                low_memory=False,
            )
            dfs.append(temp)

trip_raw = pd.concat(dfs, ignore_index=True)

display(trip_raw.head())

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,984F632114B98410,electric_bike,2024-07-11 13:32:52.359,2024-07-11 13:41:04.825,9 Ave & W 18 St,6190.08,9 Ave & W 33 St,6492.08,40.743174,-74.003664,40.752568,-73.996765,casual
1,9B0E546FDB460C0E,electric_bike,2024-07-13 13:18:42.179,2024-07-13 13:22:46.631,W 42 St & 6 Ave,6517.08,W 49 St & 8 Ave,6747.06,40.754920,-73.984550,40.762272,-73.987882,member
2,6B939445A283D985,classic_bike,2024-07-08 20:34:27.848,2024-07-08 20:41:46.350,8 Ave & W 52 St,6816.07,9 Ave & W 33 St,6492.08,40.763707,-73.985162,40.752568,-73.996765,member
3,49444E058931E427,electric_bike,2024-07-14 15:42:44.695,2024-07-14 15:55:54.771,W 120 St & Claremont Ave,7745.07,W 78 St & Broadway,7311.07,40.810949,-73.963400,40.783400,-73.980931,casual
4,74033CB639411DA0,classic_bike,2024-07-09 08:23:38.797,2024-07-09 08:28:48.647,W 42 St & 6 Ave,6517.08,W 49 St & 8 Ave,6747.06,40.754920,-73.984550,40.762272,-73.987882,member


## 4 转换时间字段并检查完整性

In [17]:
trip_base = trip_raw.copy()

trip_base["started_at"] = pd.to_datetime(trip_base["started_at"], errors="coerce")
trip_base["ended_at"] = pd.to_datetime(trip_base["ended_at"], errors="coerce")

trip_base["ride_duration_min"] = (
    trip_base["ended_at"] - trip_base["started_at"]
).dt.total_seconds() / 60

display(
    trip_base[
        [
            "ride_id",
            "started_at",
            "ended_at",
            "ride_duration_min",
        ]
    ].head()
)

time_check = pd.Series(
    {
        "rows": len(trip_base),
        "missing_started_at": trip_base["started_at"].isna().sum(),
        "missing_ended_at": trip_base["ended_at"].isna().sum(),
        "missing_ride_duration_min": trip_base["ride_duration_min"].isna().sum(),
        "non_positive_duration": (trip_base["ride_duration_min"] <= 0).sum(),
        "over_24h_duration": (trip_base["ride_duration_min"] > 24 * 60).sum(),
    }
).to_frame("value")

display(time_check)

,ride_id,started_at,ended_at,ride_duration_min
0,984F632114B98410,2024-07-11 13:32:52.359,2024-07-11 13:41:04.825,8.207767
1,9B0E546FDB460C0E,2024-07-13 13:18:42.179,2024-07-13 13:22:46.631,4.074200
2,6B939445A283D985,2024-07-08 20:34:27.848,2024-07-08 20:41:46.350,7.308367
3,49444E058931E427,2024-07-14 15:42:44.695,2024-07-14 15:55:54.771,13.167933
4,74033CB639411DA0,2024-07-09 08:23:38.797,2024-07-09 08:28:48.647,5.164167


,value
rows,4722896
missing_started_at,0
missing_ended_at,0
missing_ride_duration_min,0
non_positive_duration,0
over_24h_duration,1294


**观察与总结**

`started_at` 和 `ended_at` 已成功转换为 datetime 类型，并基于两者计算出 `ride_duration_min`。

时间字段完整性检查显示，开始时间和结束时间都没有缺失数据，没有小于 0 的异常骑行时长，存在 1294 条大于 24 小时的骑行记录。后续将基于 `started_at` 限定 2024 年 7 月自然月，并 剔除异常时长记录。

## 5 过滤为2024年7月自然月数据

01 中发现，虽然原始文件名为 202407，但数据中包含少量 6 月 30 日边界记录。为了保证后续日期趋势和月度口径清晰，本段将数据范围限定为 2024 年 7 月自然月。

In [18]:
july_start = pd.Timestamp("2024-07-01")
aug_start = pd.Timestamp("2024-08-01")

before_filter_rows = len(trip_base)

trip_base = trip_base[
    (trip_base["started_at"] >= july_start)
    & (trip_base["started_at"] < aug_start)
].copy()

after_filter_rows = len(trip_base)

print("过滤前记录数:", before_filter_rows)
print("过滤后记录数:", after_filter_rows)
print("过滤掉记录数:", before_filter_rows - after_filter_rows)

过滤前记录数: 4722896
过滤后记录数: 4722196
过滤掉记录数: 700


**观察与总结**

过滤前共有 4722896 条记录，过滤为 2024 年 7 月自然月后剩余 4722196 条记录，过滤掉 700 条记录。

这说明原始 202407 文件中确实包含少量 6 月 30 日边界记录。为了保证后续日期趋势和月度口径一致，本项目后续分析统一使用 `2024-07-01 <= started_at < 2024-08-01` 的自然月口径。

## 6 处理异常骑行时长

01 中发现骑行时长不存在小于等于 0 的记录，但存在少量超过 24 小时的极长骑行记录。此类记录可能会拉高平均骑行时长，因此本段剔除骑行时长小于等于 0 或超过 24 小时的记录。

In [19]:
before_duration_filter_rows = len(trip_base)

trip_base = trip_base[
    (trip_base["ride_duration_min"] > 0)
    & (trip_base["ride_duration_min"] <= 24 * 60)
].copy()

after_duration_filter_rows = len(trip_base)

print("时长过滤前记录数:", before_duration_filter_rows)
print("时长过滤后记录数:", after_duration_filter_rows)
print("过滤掉记录数:", before_duration_filter_rows - after_duration_filter_rows)

时长过滤前记录数: 4722196
时长过滤后记录数: 4720941
过滤掉记录数: 1255


**观察与总结**

时长过滤前共有 4722196 条记录，剔除骑行时长小于等于 0 或超过 24 小时的记录后，剩余 4720941 条记录，共过滤掉 1255 条异常时长记录。

异常时长记录占比较低，约为 0.03%，对整体骑行需求规模影响很小。但这类极端记录会拉高平均骑行时长，因此在构建 `trip_base` 时剔除是合理的。后续骑行时长相关指标均基于清洗后的有效骑行记录计算。

## 7 构造日期、小时和星期字段

本段基于骑行开始时间构造日期、月份、小时、星期和周末标记字段。这些字段是后续分析日期趋势、小时高峰、工作日与周末差异的基础。

In [20]:
trip_base["start_date"] = trip_base["started_at"].dt.date
trip_base["start_month"] = trip_base["started_at"].dt.to_period("M").astype(str)
trip_base["start_hour"] = trip_base["started_at"].dt.hour
trip_base["day_of_week"] = trip_base["started_at"].dt.day_name()
trip_base["day_of_week_num"] = trip_base["started_at"].dt.dayofweek
trip_base["is_weekend"] = trip_base["day_of_week_num"].isin([5, 6])

display(
    trip_base[
        [
            "started_at",
            "start_date",
            "start_month",
            "start_hour",
            "day_of_week",
            "day_of_week_num",
            "is_weekend",
        ]
    ].head()
)

,started_at,start_date,start_month,start_hour,day_of_week,day_of_week_num,is_weekend
0,2024-07-11 13:32:52.359,2024-07-11,2024-07,13,Thursday,3,False
1,2024-07-13 13:18:42.179,2024-07-13,2024-07,13,Saturday,5,True
2,2024-07-08 20:34:27.848,2024-07-08,2024-07,20,Monday,0,False
3,2024-07-14 15:42:44.695,2024-07-14,2024-07,15,Sunday,6,True
4,2024-07-09 08:23:38.797,2024-07-09,2024-07,8,Tuesday,1,False


**观察与总结**

本段已基于骑行开始时间构造出日期、月份、小时、星期序号、星期名称和周末标记字段。

这些字段是后续需求规律分析的基础：`start_date` 用于分析每日骑行趋势，`start_hour` 用于分析小时高峰，`day_of_week` 和 `is_weekend` 用于比较工作日与周末的骑行差异。

## 8 构造骑行时段字段

为了更直观地分析共享单车需求是否具有通勤高峰特征，本段基于骑行开始小时将一天划分为深夜、早高峰、日间、晚高峰和夜间五个时段。

In [21]:
time_period_bins = [0, 6, 10, 16, 20, 24]

time_period_labels = [
    "late_night",
    "morning_peak",
    "daytime",
    "evening_peak",
    "night",
]

trip_base["time_period"] = pd.cut(
    trip_base["start_hour"],
    bins=time_period_bins,
    labels=time_period_labels,
    right=False,
)

time_period_summary = (
    trip_base["time_period"]
    .value_counts(sort=False)
    .rename_axis("time_period")
    .reset_index(name="rides")
)

time_period_summary["ride_share"] = (
    time_period_summary["rides"] / time_period_summary["rides"].sum()
)

display(time_period_summary)

,time_period,rides,ride_share
0,late_night,217809,0.046137
1,morning_peak,740718,0.156900
2,daytime,1463731,0.310051
3,evening_peak,1551834,0.328713
4,night,746849,0.158199


**观察与总结**

从时段分布看，晚高峰骑行次数最多，约155.18万次，占比约32.87%；日间骑行约146.37万次，占比约31.01%。早高峰和夜间占比分别约15.69%和15.82%，深夜占比约4.61%。

这说明 2024 年 7 月 Citi Bike 骑行需求并不只集中在早高峰，晚高峰和日间也是主要使用时段。后续可以进一步结合会员和临时用户类型，判断不同用户群体是否存在通勤型和休闲型使用差异。

## 9 构造骑行时长分组字段

本段将连续的骑行时长划分为若干区间，便于后续比较短途、中等时长和长时长骑行的占比，也方便分析会员和临时用户的使用差异。

In [22]:
duration_bins = [0, 5, 10, 20, 30, 60, 24 * 60]

duration_labels = [
    "0_5min",
    "5_10min",
    "10_20min",
    "20_30min",
    "30_60min",
    "60min_plus",
]

trip_base["duration_group"] = pd.cut(
    trip_base["ride_duration_min"],
    bins=duration_bins,
    labels=duration_labels,
    right=True,
)

duration_group_summary = (
    trip_base["duration_group"]
    .value_counts(sort=False)
    .rename_axis("duration_group")
    .reset_index(name="rides")
)

duration_group_summary["ride_share"] = (
    duration_group_summary["rides"] / duration_group_summary["rides"].sum()
)

display(duration_group_summary)

,duration_group,rides,ride_share
0,0_5min,1003306,0.212522
1,5_10min,1437129,0.304416
2,10_20min,1387077,0.293814
3,20_30min,512698,0.108601
4,30_60min,313957,0.066503
5,60min_plus,66774,0.014144


**观察与总结**

从骑行时长分组看，5 到 10 分钟骑行最多，占比约30.44%；10 到 20 分钟占比约29.38%；0 到 5 分钟占比约21.25%。也就是说，20 分钟以内的短途骑行合计占比约81.08%。

30 分钟以上骑行占比较低，其中30 到 60 分钟占比约6.65%，60 分钟以上占比约1.41%。这说明 Citi Bike 主要服务短途城市出行场景，后续分析会员和临时用户差异时，可以重点比较两类用户在短途和长时长骑行上的结构差异。

## 10 构造不同分析口径的样本标记

不同分析目标对字段完整性的要求不同。整体需求分析主要依赖时间和用户类型字段；站点失衡分析需要起点站和终点站完整；地图分析则需要经纬度字段完整。

因此，本段不因为站点或经纬度缺失直接删除整行，而是新增样本标记字段，后续在不同分析场景中按需筛选。

In [23]:
trip_base["has_start_station"] = (
    trip_base["start_station_id"].notna()
    & trip_base["start_station_name"].notna()
)

trip_base["has_end_station"] = (
    trip_base["end_station_id"].notna()
    & trip_base["end_station_name"].notna()
)

trip_base["has_station_pair"] = (
    trip_base["has_start_station"]
    & trip_base["has_end_station"]
)

trip_base["has_start_location"] = (
    trip_base["start_lat"].notna()
    & trip_base["start_lng"].notna()
)

trip_base["has_end_location"] = (
    trip_base["end_lat"].notna()
    & trip_base["end_lng"].notna()
)

trip_base["has_location_pair"] = (
    trip_base["has_start_location"]
    & trip_base["has_end_location"]
)

trip_base["is_station_sample"] = trip_base["has_station_pair"]
trip_base["is_map_sample"] = trip_base["has_location_pair"]

sample_summary = pd.Series(
    {
        "trip_base_rows": len(trip_base),
        "station_sample_rows": trip_base["is_station_sample"].sum(),
        "map_sample_rows": trip_base["is_map_sample"].sum(),
    }
).to_frame("value")

display(sample_summary)

,value
trip_base_rows,4720941
station_sample_rows,4706967
map_sample_rows,4706967


**观察与总结**

清洗后的 `trip_base` 共有 4720941 条记录，整体需求分析样本同样为 4720941 条，说明所有有效骑行记录都可以用于日期趋势、小时趋势和用户类型分析。

站点分析样本和地图分析样本均为 4706967 条，占清洗后样本约99.70%。约0.30%的记录存在起终点站点或经纬度不完整问题，因此后续站点失衡和地图可视化需要使用 `is_station_sample` 或 `is_map_sample` 进行筛选。

这种处理方式避免了为了站点分析而提前删除全量记录，使整体需求分析和站点分析可以分别使用合适的样本口径。

## 11 检查最终分析表粒度和核心指标

本段检查清洗后 `trip_base` 的记录数、骑行ID唯一性、时间范围和骑行时长概况，确认最终分析表仍保持“一行一次骑行事件”的基础粒度。

In [24]:
trip_base_check = pd.Series(
    {
        "rows": len(trip_base),
        "unique_ride_id": trip_base["ride_id"].nunique(),
        "duplicated_ride_id": len(trip_base) - trip_base["ride_id"].nunique(),
        "min_started_at": trip_base["started_at"].min(),
        "max_started_at": trip_base["started_at"].max(),
        "avg_duration_min": trip_base["ride_duration_min"].mean(),
        "median_duration_min": trip_base["ride_duration_min"].median(),
    }
).to_frame("value")

display(trip_base_check)

,value
rows,4720941
unique_ride_id,4720941
duplicated_ride_id,0
min_started_at,2024-07-01 00:00:00.191000
max_started_at,2024-07-31 23:57:41.895000
avg_duration_min,14.13168
median_duration_min,9.654083


**观察与总结**

最终 `trip_base` 共有 4720941 条记录，`ride_id` 唯一值数量同样为 4720941，重复 `ride_id` 数量为 0，说明清洗后的分析表仍保持“一行一次骑行事件”的基础粒度。

时间范围为 2024 年 7 月 1 日 00:00:00 至 2024 年 7 月 31 日 23:57:41，说明自然月过滤已生效。清洗后平均骑行时长约14.13分钟，中位数约9.65分钟，平均值仍高于中位数，说明骑行时长分布依然右偏，但超过24小时的极端值已经被剔除。

## 12 导出清洗后的骑行事件分析表

本段将清洗和字段构建后的 `trip_base` 导出为 CSV 文件，作为后续需求规律分析、站点失衡分析、用户类型差异分析和 Power BI 看板的数据基础。

In [25]:
output_path = DATA_CLEAN_DIR / "trip_base_202407.csv"

trip_base.to_csv(output_path, index=False)

print("已导出:", output_path)
print("导出记录数:", len(trip_base))

已导出: ..\data_clean\trip_base_202407.csv
导出记录数: 4720941


## 总结

本 Notebook 基于 2024 年 7 月 Citi Bike 原始月度骑行数据，构建了清洗后的事件级分析表 `trip_base_202407.csv`。

首先，原始压缩包中包含 5 个 CSV 文件，本 Notebook 使用统一读取逻辑将其合并为原始骑行表 `trip_raw`。读取过程中将骑行ID、站点ID、站点名称、车型和用户类型等字段指定为字符串，避免站点ID被误识别为数值字段。

随后，本 Notebook 将开始时间和结束时间转换为 datetime 类型，并计算每次骑行的持续时长。由于 01 中发现原始文件包含少量 6 月 30 日边界记录，本 Notebook 将分析范围限定为 2024 年 7 月自然月。自然月过滤后，记录数从 4722896 条变为 4722196 条，共过滤掉 700 条边界记录。

在异常时长处理方面，本 Notebook 剔除了骑行时长小于等于 0 或超过 24 小时的记录。过滤后最终保留 4720941 条有效骑行记录。异常时长记录占比较低，对整体骑行需求规模影响较小，但剔除后可以减少极端值对平均骑行时长和用户行为分析的影响。

构建后的 `trip_base` 仍保持“一行一次骑行事件”的基础粒度。最终表中 `ride_id` 唯一值数量与总记录数一致，重复 `ride_id` 数量为 0，说明该表可以作为后续事件级分析基础。

字段构建方面，本 Notebook 新增了日期、月份、小时、星期、周末标记、骑行时段和骑行时长分组字段。初步结果显示，晚高峰和日间是骑行需求最集中的时段；20 分钟以内骑行占比约81.08%，说明 Citi Bike 主要服务短途城市出行场景。

样本口径方面，整体需求分析使用全部有效骑行记录；站点失衡分析使用 `is_station_sample` 筛选起终点站点字段完整的记录；地图可视化使用 `is_map_sample` 筛选起终点经纬度完整的记录。这样可以避免不同分析任务之间口径混乱。

下一步将在 `03_demand_pattern_analysis.ipynb` 中，基于 `trip_base_202407.csv` 分析每日骑行趋势、小时高峰、星期差异、工作日与周末差异，并进一步判断 Citi Bike 需求是否具有明显通勤高峰特征。